# Disaster Image Classification — Training and Evaluation

This notebook contains the complete training and evaluation workflow for four experimental configurations:

1. **CNN**
2. **CNN + Fine Tuning**
3. **Inception ResNet V2**
4. **Inception ResNet V2 + Fine Tuning**

The workflow includes dataset preparation, corrupted-image validation, train/validation/test splitting, image preprocessing and augmentation, model training, model saving, and evaluation using:

- Accuracy
- Sensitivity / Recall
- Specificity
- Precision
- F1-Score
- Confusion Matrix

The dataset is downloaded automatically from Kaggle:
`varpit94/disaster-images-dataset`


## 1. Setup

In [ ]:
%pip install -q kagglehub

from pathlib import Path
import gc
import random
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from PIL import Image, ImageFile
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split

from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger
from tensorflow.keras.applications import InceptionResNetV2

import kagglehub
from google.colab import drive

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ImageFile.LOAD_TRUNCATED_IMAGES = False

print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices("GPU"))

## 2. Download dataset dan konfigurasi

In [ ]:
# download dataset
drive.mount("/content/drive")

dataset_root = Path(
    kagglehub.dataset_download(
        "varpit94/disaster-images-dataset"
    )
)

EXPECTED_CLASSES = {
    "Damaged_Infrastructure",
    "Fire_Disaster",
    "Human_Damage",
    "Land_Disaster",
    "Non_Damage",
    "Water_Disaster",
}

# mencari folder yang memiliki keenam folder kelas.
DATASET_DIR = None

candidate_dirs = [dataset_root]
candidate_dirs.extend(
    p for p in dataset_root.rglob("*")
    if p.is_dir()
)

for folder in candidate_dirs:
    try:
        child_dirs = {
            p.name
            for p in folder.iterdir()
            if p.is_dir()
        }
    except OSError:
        continue

    if EXPECTED_CLASSES.issubset(child_dirs):
        DATASET_DIR = folder
        break

if DATASET_DIR is None:
    raise FileNotFoundError(
        "Folder yang berisi 6 kelas dataset tidak ditemukan."
    )

# semua output disimpan ke Google Drive.
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Disaster"
)

MODEL_DIR = OUTPUT_DIR / "models"
HISTORY_DIR = OUTPUT_DIR / "history"
LOG_DIR = OUTPUT_DIR / "logs"

for folder in [OUTPUT_DIR, MODEL_DIR, HISTORY_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# konfigurasi
IMG_SIZE = (299, 299)
BATCH_SIZE = 32

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

ROTATION_RANGE = 15
ZOOM_RANGE = 0.10
HORIZONTAL_FLIP = True

BASE_EPOCHS = 15
CNN_FINE_TUNE_EPOCHS = 5
IRV2_FINE_TUNE_EPOCHS = 10

BASE_LR = 1e-4
FINE_TUNE_LR = 1e-5

print()
print("Dataset :", DATASET_DIR)
print("Output  :", OUTPUT_DIR)

print("\nKelas:")
for class_name in sorted(EXPECTED_CLASSES):
    print("✓", class_name)

## 3. Membaca dataset

In [ ]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

records = []

for class_name in sorted(EXPECTED_CLASSES):
    class_dir = DATASET_DIR / class_name

    for image_path in class_dir.rglob("*"):
        if (
            image_path.is_file()
            and image_path.suffix.lower() in IMAGE_EXTENSIONS
        ):
            records.append(
                {
                    "filepath": str(image_path),
                    "label": class_name,
                }
            )

df = pd.DataFrame(records)

if df.empty:
    raise RuntimeError(
        "Tidak ada file gambar yang ditemukan."
    )

display(
    df["label"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

print(f"\nTotal file gambar: {len(df):,}")

## 4. Validasi file gambar rusak

In [ ]:
valid_records = []
corrupt_records = []

for row in tqdm(
    df.itertuples(index=False),
    total=len(df),
    desc="Memeriksa gambar",
):
    image_path = Path(row.filepath)

    try:
        # img.load() membaca data piksel penuh, sehingga file truncated dapat terdeteksi.
        with Image.open(image_path) as img:
            img.load()
            img.convert("RGB")

        valid_records.append(
            {
                "filepath": row.filepath,
                "label": row.label,
            }
        )

    except Exception as e:
        corrupt_records.append(
            {
                "filepath": row.filepath,
                "label": row.label,
                "error": str(e),
            }
        )

corrupt_df = pd.DataFrame(corrupt_records)
df = pd.DataFrame(valid_records).reset_index(drop=True)

print()
print(f"File valid : {len(df):,}")
print(f"File rusak : {len(corrupt_df):,}")

if not corrupt_df.empty:
    display(corrupt_df)

    corrupt_path = LOG_DIR / "corrupt_images.csv"
    corrupt_df.to_csv(
        corrupt_path,
        index=False,
    )

    print(
        "\nDaftar file rusak:",
        corrupt_path,
    )

display(
    df["label"]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

CLASS_NAMES = sorted(df["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)

if NUM_CLASSES != 6:
    raise RuntimeError(
        f"Jumlah kelas seharusnya 6, tetapi ditemukan {NUM_CLASSES}."
    )

## 5. Split train / validation / test

In [ ]:
if not np.isclose(
    TRAIN_RATIO + VAL_RATIO + TEST_RATIO,
    1.0,
):
    raise ValueError(
        "TRAIN_RATIO + VAL_RATIO + TEST_RATIO harus = 1.0"
    )

train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df["label"],
    random_state=SEED,
)

relative_test_ratio = (
    TEST_RATIO
    / (VAL_RATIO + TEST_RATIO)
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_ratio,
    stratify=temp_df["label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Training   : {len(train_df):,}")
print(f"Validation : {len(val_df):,}")
print(f"Testing    : {len(test_df):,}")

split_summary = pd.concat(
    [
        train_df["label"]
        .value_counts()
        .rename("train"),

        val_df["label"]
        .value_counts()
        .rename("validation"),

        test_df["label"]
        .value_counts()
        .rename("test"),
    ],
    axis=1,
).fillna(0).astype(int)

print()
display(split_summary.sort_index())

## 6. Data generator

In [ ]:
# augmentasi hanya untuk training
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=ROTATION_RANGE,
    zoom_range=ZOOM_RANGE,
    horizontal_flip=HORIZONTAL_FLIP,
)

# validation dan test hanya normalisasi
eval_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED,
)

val_gen = eval_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

test_gen = eval_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col="filepath",
    y_col="label",
    classes=CLASS_NAMES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

print("\nClass indices:")
print(train_gen.class_indices)

## 7. Fungsi evaluasi

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

import matplotlib.pyplot as plt

METRICS_DIR = OUTPUT_DIR / "metrics"
PLOTS_DIR = OUTPUT_DIR / "plots"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

def macro_specificity(cm):
    specificities = []

    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0.0
        )

        specificities.append(specificity)

    return float(np.mean(specificities))

def draw_confusion_matrix(cm, class_names, model_name, save_path):
    fig, ax = plt.subplots(figsize=(10, 8), facecolor="white")

    image = ax.imshow(cm, cmap="Blues")

    ax.set_title(f"Confusion Matrix — {model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))

    ax.set_xticklabels(
        class_names,
        rotation=90,
        ha="center",
        va="top",
    )
    ax.set_yticklabels(class_names)

    ax.tick_params(axis="x", pad=10)

    threshold = cm.max() / 2.0 if cm.size else 0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_color = "white" if cm[i, j] > threshold else "black"
            ax.text(
                j,
                i,
                str(cm[i, j]),
                ha="center",
                va="center",
                color=text_color,
            )

    fig.colorbar(image, ax=ax)

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=200,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.show()

def evaluate_model(model, generator, model_name):
    generator.reset()

    probabilities = model.predict(
        generator,
        verbose=1,
    )

    y_pred = np.argmax(
        probabilities,
        axis=1,
    )

    y_true = generator.classes

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=np.arange(NUM_CLASSES),
            average="macro",
            zero_division=0,
        )
    )

    metrics = {
        "model": model_name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "sensitivity": float(recall),
        "specificity": float(macro_specificity(cm)),
        "precision": float(precision),
        "f1_score": float(f1),
    }

    print()
    print(f"Accuracy    : {metrics['accuracy']:.4f}")
    print(f"Sensitivity : {metrics['sensitivity']:.4f}")
    print(f"Specificity : {metrics['specificity']:.4f}")
    print(f"Precision   : {metrics['precision']:.4f}")
    print(f"F1-Score    : {metrics['f1_score']:.4f}")

    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=np.arange(NUM_CLASSES),
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        )
    ).transpose()

    report_df.to_csv(
        METRICS_DIR / f"{model_name}_classification_report.csv"
    )

    cm_df = pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    )

    cm_df.to_csv(
        METRICS_DIR / f"{model_name}_confusion_matrix.csv"
    )

    pd.DataFrame([metrics]).to_csv(
        METRICS_DIR / f"{model_name}_metrics.csv",
        index=False,
    )

    draw_confusion_matrix(
        cm=cm,
        class_names=CLASS_NAMES,
        model_name=model_name,
        save_path=PLOTS_DIR / f"{model_name}_confusion_matrix.png",
    )

    return metrics

def history_dataframe(history, start_epoch=1):
    df_history = pd.DataFrame(history.history)

    df_history.insert(
        0,
        "epoch",
        np.arange(start_epoch, start_epoch + len(df_history)),
    )

    return df_history

def save_combined_history(base_history, fine_history, model_name):
    base_df = history_dataframe(
        base_history,
        start_epoch=1,
    )

    fine_df = history_dataframe(
        fine_history,
        start_epoch=(len(base_df) + 1),
    )

    combined = pd.concat(
        [base_df, fine_df],
        ignore_index=True,
    )

    combined.to_csv(
        HISTORY_DIR / f"{model_name}_combined_history.csv",
        index=False,
    )

    plt.figure(figsize=(10, 5))
    plt.plot(
        combined["epoch"],
        combined["accuracy"],
        label="Training Accuracy",
    )
    plt.plot(
        combined["epoch"],
        combined["val_accuracy"],
        label="Validation Accuracy",
    )
    plt.axvline(
        x=len(base_df),
        linestyle="--",
        label="Fine Tuning",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} — Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / f"{model_name}_accuracy.png",
        dpi=200,
        facecolor="white",
    )
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(
        combined["epoch"],
        combined["loss"],
        label="Training Loss",
    )
    plt.plot(
        combined["epoch"],
        combined["val_loss"],
        label="Validation Loss",
    )
    plt.axvline(
        x=len(base_df),
        linestyle="--",
        label="Fine Tuning",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} — Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        PLOTS_DIR / f"{model_name}_loss.png",
        dpi=200,
        facecolor="white",
    )
    plt.show()

    return combined

# 8. Model 1 — CNN

In [ ]:
def build_cnn(num_classes):
    model = Sequential(
        [
            Input(
                shape=(
                    *IMG_SIZE,
                    3,
                )
            ),

            Conv2D(
                32,
                (3, 3),
                activation="relu",
                padding="same",
            ),
            MaxPooling2D(
                (2, 2)
            ),

            Conv2D(
                64,
                (3, 3),
                activation="relu",
                padding="same",
            ),
            MaxPooling2D(
                (2, 2)
            ),

            Conv2D(
                128,
                (3, 3),
                activation="relu",
                padding="same",
            ),
            MaxPooling2D(
                (2, 2)
            ),

            Conv2D(
                256,
                (3, 3),
                activation="relu",
                padding="same",
            ),
            MaxPooling2D(
                (2, 2)
            ),

            Flatten(),

            Dense(
                256,
                activation="relu",
            ),

            Dropout(
                0.5
            ),

            Dense(
                num_classes,
                activation="softmax",
            ),
        ],
        name="CNN",
    )

    model.compile(
        optimizer=Adam(
            learning_rate=BASE_LR
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model

cnn_model = build_cnn(
    NUM_CLASSES
)

cnn_model.summary()

In [ ]:
CNN_MODEL_PATH = (
    MODEL_DIR
    / "01_cnn.keras"
)

cnn_callbacks = [
    ModelCheckpoint(
        filepath=CNN_MODEL_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    CSVLogger(
        HISTORY_DIR
        / "01_cnn_history.csv"
    ),
]

cnn_history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=BASE_EPOCHS,
    callbacks=cnn_callbacks,
)

cnn_model = (
    tf.keras.models.load_model(
        CNN_MODEL_PATH
    )
)

cnn_metrics = evaluate_model(
    cnn_model,
    test_gen,
    "cnn",
)

print()
print("Model tersimpan:")
print(CNN_MODEL_PATH)

# 9. Model 2 — CNN + Fine Tuning

In [ ]:
CNN_FINE_MODEL_PATH = (
    MODEL_DIR
    / "02_cnn_fine_tuning.keras"
)

cnn_model.compile(
    optimizer=Adam(
        learning_rate=FINE_TUNE_LR
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_fine_callbacks = [
    ModelCheckpoint(
        filepath=CNN_FINE_MODEL_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    CSVLogger(
        HISTORY_DIR
        / "02_cnn_fine_tuning_history.csv"
    ),
]

cnn_fine_history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    initial_epoch=BASE_EPOCHS,
    epochs=(
        BASE_EPOCHS
        + CNN_FINE_TUNE_EPOCHS
    ),
    callbacks=cnn_fine_callbacks,
)

cnn_model = (
    tf.keras.models.load_model(
        CNN_FINE_MODEL_PATH
    )
)

cnn_fine_metrics = (
    evaluate_model(
        cnn_model,
        test_gen,
        "cnn_fine_tuning",
    )
)

cnn_combined_history = (
    save_combined_history(
        cnn_history,
        cnn_fine_history,
        "cnn",
    )
)

print()
print("Model tersimpan:")
print(CNN_FINE_MODEL_PATH)

In [ ]:
del cnn_model

tf.keras.backend.clear_session()
gc.collect()

print("Memori CNN dibersihkan.")

# 10. Model 3 — Inception ResNet V2

In [ ]:
def build_inception_resnet_v2(
    num_classes,
):
    base_model = InceptionResNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(
            *IMG_SIZE,
            3,
        ),
    )

    base_model.trainable = False

    inputs = Input(
        shape=(
            *IMG_SIZE,
            3,
        )
    )

    x = base_model(
        inputs,
        training=False,
    )

    x = GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = Dropout(
        0.2,
        name="dropout",
    )(x)

    outputs = Dense(
        num_classes,
        activation="softmax",
        name="classifier",
    )(x)

    model = Model(
        inputs,
        outputs,
        name="InceptionResNetV2_Disaster",
    )

    model.compile(
        optimizer=Adam(
            learning_rate=BASE_LR
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model, base_model


irv2_model, irv2_base = (
    build_inception_resnet_v2(
        NUM_CLASSES
    )
)

irv2_model.summary()

In [ ]:
IRV2_MODEL_PATH = (
    MODEL_DIR
    / "03_inception_resnet_v2.keras"
)

irv2_callbacks = [
    ModelCheckpoint(
        filepath=IRV2_MODEL_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    CSVLogger(
        HISTORY_DIR
        / "03_inception_resnet_v2_history.csv"
    ),
]

irv2_history = irv2_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=BASE_EPOCHS,
    callbacks=irv2_callbacks,
)

irv2_model = (
    tf.keras.models.load_model(
        IRV2_MODEL_PATH
    )
)

irv2_metrics = (
    evaluate_model(
        irv2_model,
        test_gen,
        "inception_resnet_v2",
    )
)

print()
print("Model tersimpan:")
print(IRV2_MODEL_PATH)

# 11. Model 4 — Inception ResNet V2 + Fine Tuning

In [ ]:
irv2_base = None

for layer in irv2_model.layers:
    if (
        isinstance(
            layer,
            tf.keras.Model,
        )
        and
        "inception_resnet"
        in layer.name.lower()
    ):
        irv2_base = layer
        break

if irv2_base is None:
    raise RuntimeError(
        "Backbone InceptionResNetV2 "
        "tidak ditemukan."
    )

irv2_base.trainable = True

for layer in irv2_base.layers[:-40]:
    layer.trainable = False

for layer in irv2_base.layers[-40:]:
    layer.trainable = True

trainable_layers = sum(
    int(layer.trainable)
    for layer
    in irv2_base.layers
)

print(
    "Total layer backbone     :",
    len(irv2_base.layers),
)

print(
    "Layer backbone trainable :",
    trainable_layers,
)

print(
    "Target fine tuning       : 40"
)

In [ ]:
IRV2_FINE_MODEL_PATH = (
    MODEL_DIR
    / "04_inception_resnet_v2_fine_tuning.keras"
)

irv2_model.compile(
    optimizer=Adam(
        learning_rate=FINE_TUNE_LR
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

irv2_fine_callbacks = [
    ModelCheckpoint(
        filepath=IRV2_FINE_MODEL_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    CSVLogger(
        HISTORY_DIR
        / "04_inception_resnet_v2_fine_tuning_history.csv"
    ),
]

irv2_fine_history = (
    irv2_model.fit(
        train_gen,
        validation_data=val_gen,
        initial_epoch=BASE_EPOCHS,
        epochs=(
            BASE_EPOCHS
            + IRV2_FINE_TUNE_EPOCHS
        ),
        callbacks=irv2_fine_callbacks,
    )
)

irv2_model = (
    tf.keras.models.load_model(
        IRV2_FINE_MODEL_PATH
    )
)

irv2_fine_metrics = (
    evaluate_model(
        irv2_model,
        test_gen,
        "inception_resnet_v2_fine_tuning",
    )
)

irv2_combined_history = (
    save_combined_history(
        irv2_history,
        irv2_fine_history,
        "inception_resnet_v2",
    )
)

print()
print("Model tersimpan:")
print(IRV2_FINE_MODEL_PATH)

## 13. Verifikasi semua output

In [ ]:
saved_models = [
    CNN_MODEL_PATH,
    CNN_FINE_MODEL_PATH,
    IRV2_MODEL_PATH,
    IRV2_FINE_MODEL_PATH,
]

for model_path in saved_models:
    if model_path.exists():
        size_mb = (
            model_path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"✓ {model_path.name:<50} "
            f"{size_mb:,.2f} MB"
        )
    else:
        print(
            f"✗ {model_path.name} "
            "TIDAK DITEMUKAN"
        )

print()
print("Model   :", MODEL_DIR)
print("Metrics :", METRICS_DIR)
print("Plots   :", PLOTS_DIR)
print("History :", HISTORY_DIR)

## Output

After all cells are executed, the main outputs are stored in:

```text
/content/drive/MyDrive/Disaster/
├── models/
│   ├── 01_cnn.keras
│   ├── 02_cnn_fine_tuning.keras
│   ├── 03_inception_resnet_v2.keras
│   └── 04_inception_resnet_v2_fine_tuning.keras
├── metrics/
│   ├── *_metrics.csv
│   ├── *_classification_report.csv
│   └── *_confusion_matrix.csv
├── plots/
│   ├── *_confusion_matrix.png
│   ├── cnn_accuracy.png
│   ├── cnn_loss.png
│   ├── inception_resnet_v2_accuracy.png
│   └── inception_resnet_v2_loss.png
├── history/
└── logs/
```

The notebook ends after training and evaluating the four model configurations.
